# 01 Data understanding

Inspect the five released TRACE-X Parquet files. Each table gets a schema and missing-value summary, the first five stored rows, numerical summaries, and any date ranges. Then check identifiers and table relationships. The last cell writes the schema report.

These checks use the released data in read-only mode. The first five rows follow file order. This is a structural review with no split-specific feature or target analysis.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import Markdown, display

# Run from either the project root or its notebooks folder.
working_directory = Path.cwd()
project_root = next(
    path for path in (working_directory, *working_directory.parents)
    if (path / "AGENTS.md").is_file() and (path / "Phase1_Dataset" / "dataset").is_dir()
)
data_dir = project_root / "Phase1_Dataset" / "dataset"
table_names = ("entities", "medicines", "batches", "fraud_events", "transactions")
paths = {name: data_dir / f"{name}.parquet" for name in table_names}
batch_size = 50_000
target = "is_fraud_ground_truth"
date_formats = {
    "entities": {"license_expiry": "%Y-%m-%d"},
    "batches": {"production_date": "%Y-%m-%d", "expiry_date": "%Y-%m-%d"},
    "transactions": {"timestamp": "%Y-%m-%dT%H:%M:%S"},
}
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | PyArrow {pa.__version__}")
print(f"Read batch size: {batch_size:,} rows")

## Table structure, samples, and ranges

The scan counts Parquet nulls and also checks floating-point columns for NaN. Numerical summaries use finite values and show the sample standard deviation. Booleans are displayed in the schema and samples. Date text is parsed in memory only to check its format and range.

In [ ]:
def count_true(values):
    """Count true values in an Arrow Boolean array, ignoring nulls."""
    return pc.sum(values).as_py() or 0


def profile_table(name):
    """Read one block at a time and keep only running totals and ranges."""
    formats = date_formats.get(name, {})
    with pq.ParquetFile(paths[name]) as parquet:
        schema = parquet.schema_arrow
        rows = parquet.metadata.num_rows
        assert set(formats).issubset(schema.names), f"Missing date field in {name}"
        missing = {field.name: [0, 0] for field in schema}
        # Count, mean, squared deviations from the mean, minimum, maximum, infinities.
        numeric = {
            field.name: (0, 0.0, 0.0, np.inf, -np.inf, 0)
            for field in schema
            if pa.types.is_integer(field.type) or pa.types.is_floating(field.type)
        }
        date_ranges = {column: [None, None, 0] for column in formats}
        first_rows = None
        scanned_rows = 0
        for block in parquet.iter_batches(batch_size=batch_size):
            if first_rows is None:
                first_rows = block.slice(0, 5).to_pandas()
            scanned_rows += len(block)
            for column, values in zip(schema.names, block.columns):
                missing[column][0] += values.null_count
                if pa.types.is_floating(values.type):
                    missing[column][1] += count_true(pc.is_nan(values))

                if column in numeric:
                    old_count, mean, squared, low, high, infinities = numeric[column]
                    raw = values.to_numpy(zero_copy_only=False)
                    infinities += int(np.count_nonzero(np.isinf(raw)))
                    finite = raw[np.isfinite(raw)]
                    if finite.size:
                        block_mean = float(finite.mean())
                        count = old_count + finite.size
                        difference = block_mean - mean
                        # Combine the block's deviations with the running total.
                        squared += float(np.square(finite - block_mean).sum())
                        squared += difference**2 * old_count * finite.size / count
                        mean += difference * finite.size / count
                        low, high = min(low, finite.min()), max(high, finite.max())
                        old_count = count
                    numeric[column] = old_count, mean, squared, low, high, infinities

                if column in formats:
                    parsed = pc.strptime(values, format=formats[column], unit="s", error_is_null=True)
                    extrema = pc.min_max(parsed).as_py()
                    start, end, invalid = date_ranges[column]
                    if extrema["min"] is not None:
                        start = extrema["min"] if start is None else min(start, extrema["min"])
                        end = extrema["max"] if end is None else max(end, extrema["max"])
                    date_ranges[column] = [start, end, invalid + parsed.null_count - values.null_count]

        assert scanned_rows == rows, f"Unexpected row count in {name}"

    columns = pd.DataFrame([
        (field.name, str(field.type), *missing[field.name], sum(missing[field.name]))
        for field in schema
    ], columns=["Column", "Stored type", "Null", "NaN", "Missing total"])
    summaries = pd.DataFrame([
        (
            column, count, low if count else None, high if count else None,
            mean if count else None, np.sqrt(squared / (count - 1)) if count > 1 else None,
            infinities,
        )
        for column, (count, mean, squared, low, high, infinities) in numeric.items()
    ], columns=["Column", "Finite count", "Min", "Max", "Mean", "Sample std", "Infinite"])
    dates = pd.DataFrame([
        (column, start.strftime(formats[column]) if start is not None else None,
         end.strftime(formats[column]) if end is not None else None, invalid)
        for column, (start, end, invalid) in date_ranges.items()
    ], columns=["Column", "Earliest", "Latest", "Invalid non-null strings"])
    return {
        "rows": rows, "columns": columns, "first_rows": first_rows,
        "numeric": summaries, "dates": dates,
    }


profiles = {}
for name in table_names:
    result = profile_table(name)
    profiles[name] = result
    display(Markdown(f"### {name}\n\nShape: {result['rows']:,} rows by {len(result['columns'])} columns."))
    display(result["columns"])
    display(Markdown("**First five stored rows**"))
    with pd.option_context("display.max_columns", None):
        display(result["first_rows"])
    if result["numeric"].empty:
        print("No numerical columns.")
    else:
        display(result["numeric"])
    if result["dates"].empty:
        print("No date or timestamp columns.")
    else:
        display(result["dates"])

## Identifiers and duplicate records

An identifier can repeat when it links to another table or describes a group. A row key must be present and unique. The project report identifies the expected keys. The checks below count distinct combinations and duplicates in the released files. A duplicate count excludes the first appearance of each value or row.

In [ ]:
small_id_columns = {
    "entities": ["entity_id", "license_number"],
    "medicines": ["medicine_id"],
    "batches": ["batch_id", "manufacturer_id", "medicine_id", "serial_number", "lot_number"],
    "fraud_events": ["batch_id"],
}
primary_keys = {
    "entities": "entity_id", "medicines": "medicine_id",
    "batches": "batch_id", "fraud_events": "batch_id",
}
identifier_rows = []
duplicate_rows = {}
small_tables = {}
for name, columns in small_id_columns.items():
    # These four tables fit in memory. The transactions file is scanned separately.
    frame = pd.read_parquet(paths[name])
    small_tables[name] = frame
    duplicate_rows[name] = int(frame.duplicated().sum())
    for column in columns:
        values = frame[column]
        distinct = values.nunique(dropna=True)
        missing_count = int(values.isna().sum())
        identifier_rows.append((name, column, distinct, missing_count, len(values) - distinct - missing_count))

for name, column in primary_keys.items():
    key = small_tables[name][column]
    assert key.notna().all() and key.is_unique, f"Invalid row key: {name}.{column}"

In [ ]:
transaction_columns = [
    "batch_id", "leg", "event_type", "batch_tx_index", "sensor_id",
    "current_hash", "previous_hash",
]
transaction_row_count = profiles["transactions"]["rows"]


def check_transaction_identifiers():
    """Count exact distinct values using compact arrays and projected Parquet scans."""
    domains = {name: set() for name in ("batch_id", "event_type", "sensor_id")}
    with pq.ParquetFile(paths["transactions"]) as parquet:
        for block in parquet.iter_batches(batch_size=batch_size, columns=list(domains)):
            for name, values in domains.items():
                values.update(pc.unique(pc.drop_null(block[name])).to_pylist())

        batch_ids = pa.array(sorted(domains["batch_id"]), type=parquet.schema_arrow.field("batch_id").type)
        event_types = pa.array(sorted(domains["event_type"]), type=parquet.schema_arrow.field("event_type").type)
        ranges = profiles["transactions"]["numeric"].set_index("Column")
        minimum_leg = int(ranges.loc["leg", "Min"])
        minimum_index = int(ranges.loc["batch_tx_index", "Min"])
        leg_count = int(ranges.loc["leg", "Max"]) - minimum_leg + 1
        index_count = int(ranges.loc["batch_tx_index", "Max"]) - minimum_index + 1
        # A mixed-radix integer stores each tuple exactly, without hash collisions.
        assert len(batch_ids) * max(leg_count * len(event_types), index_count) <= np.iinfo(np.int64).max
        event_keys = np.empty(transaction_row_count, dtype=np.int64)
        index_keys = np.empty(transaction_row_count, dtype=np.int64)
        # Store 32 binary bytes per digest instead of retaining hexadecimal strings.
        digest_buffers = {
            name: np.empty(transaction_row_count, dtype="V32")
            for name in ("current_hash", "previous_hash")
        }
        event_size = index_size = 0
        digest_sizes = dict.fromkeys(digest_buffers, 0)
        for block in parquet.iter_batches(batch_size=batch_size, columns=transaction_columns):
            batch_codes = pc.index_in(block["batch_id"], value_set=batch_ids).to_numpy(zero_copy_only=False)
            event_codes = pc.index_in(block["event_type"], value_set=event_types).to_numpy(zero_copy_only=False)
            legs = block["leg"].to_numpy(zero_copy_only=False)
            indexes = block["batch_tx_index"].to_numpy(zero_copy_only=False)

            present = np.isfinite(batch_codes) & np.isfinite(legs) & np.isfinite(event_codes)
            event_size_next = event_size + int(present.sum())
            event_keys[event_size:event_size_next] = (
                (batch_codes[present].astype(np.int64) * leg_count + legs[present].astype(np.int64) - minimum_leg)
                * len(event_types) + event_codes[present].astype(np.int64)
            )
            event_size = event_size_next

            present = np.isfinite(batch_codes) & np.isfinite(indexes)
            index_size_next = index_size + int(present.sum())
            index_keys[index_size:index_size_next] = (
                batch_codes[present].astype(np.int64) * index_count
                + indexes[present].astype(np.int64) - minimum_index
            )
            index_size = index_size_next

            for name, buffer in digest_buffers.items():
                values = pc.drop_null(block[name])
                assert pc.all(pc.match_substring_regex(values, "^[0-9a-f]{64}$")).as_py() is not False
                packed = np.frombuffer(bytes.fromhex("".join(values.to_pylist())), dtype="V32")
                size = digest_sizes[name]
                buffer[size:size + len(packed)] = packed
                digest_sizes[name] += len(packed)

    # Sorting each compact array puts repeated values next to each other.
    summaries = []
    for label, values in [
        ("(batch_id, leg, event_type)", event_keys[:event_size]),
        ("(batch_id, batch_tx_index)", index_keys[:index_size]),
        *((name, values[:digest_sizes[name]]) for name, values in digest_buffers.items()),
    ]:
        values.sort()
        repeated = int(np.count_nonzero(values[1:] == values[:-1]))
        summaries.append(("transactions", label, len(values) - repeated, transaction_row_count - len(values), repeated))

    current = digest_buffers["current_hash"][:digest_sizes["current_hash"]]
    previous = digest_buffers["previous_hash"][:digest_sizes["previous_hash"]]
    zero_digest = np.zeros(1, dtype="V32")[0]
    roots = unmatched = self_reference_values = 0
    for start in range(0, len(previous), batch_size):
        chunk = previous[start:start + batch_size]
        roots += int(np.count_nonzero(chunk == zero_digest))
        references = chunk[chunk != zero_digest]
        positions = np.searchsorted(current, references)
        within = positions < len(current)
        matched = int(np.count_nonzero(current[positions[within]] == references[within]))
        self_reference_values += matched
        unmatched += len(references) - matched

    missing = profiles["transactions"]["columns"].set_index("Column")["Missing total"]
    for name in ("batch_id", "sensor_id"):
        distinct = len(domains[name])
        absent = int(missing[name])
        summaries.append(("transactions", name, distinct, absent, transaction_row_count - distinct - absent))
    return summaries, domains["batch_id"], (roots, self_reference_values, unmatched)


transaction_ids, referenced_batch_ids, ledger_counts = check_transaction_identifiers()
identifier_rows.extend(transaction_ids)
identifiers = pd.DataFrame(
    identifier_rows, columns=["Table", "Identifier", "Distinct non-null", "Missing rows", "Repeated rows"]
)
display(identifiers)

# If a key is unique, two full rows cannot be identical: their keys would also match.
composite_keys = identifiers.loc[identifiers["Identifier"].str.startswith("(")]
assert len(composite_keys) == 2
assert composite_keys[["Missing rows", "Repeated rows"]].eq(0).all().all()
duplicate_rows["transactions"] = 0
duplicates = pd.DataFrame(duplicate_rows.items(), columns=["Table", "Duplicate records"])
display(duplicates)

## Foreign keys, relationships, and target

Compare child IDs with parent IDs using membership checks. Matching IDs establishes a relationship without building a merged table. The hash check is separate because it refers to another row in the same transactions table.

In [ ]:
relationship_specs = [
    ("batches.medicine_id", "medicines.medicine_id", "many-to-one",
     small_tables["batches"]["medicine_id"], set(small_tables["medicines"]["medicine_id"])),
    ("batches.manufacturer_id", "entities.entity_id (manufacturers)", "many-to-one",
     small_tables["batches"]["manufacturer_id"],
     set(small_tables["entities"].loc[small_tables["entities"]["entity_type"].eq("manufacturer"), "entity_id"])),
    ("fraud_events.batch_id", "batches.batch_id", "one-to-one",
     small_tables["fraud_events"]["batch_id"], set(small_tables["batches"]["batch_id"])),
]
relationship_rows = []
for child, parent, relationship, values, parent_ids in relationship_specs:
    child_ids = set(values.dropna())
    relationship_rows.append((
        child, parent, relationship, len(values), int(values.isna().sum()),
        int((values.notna() & ~values.isin(parent_ids)).sum()), len(parent_ids - child_ids),
    ))

parent_ids = set(small_tables["batches"]["batch_id"])
missing_batch_ids = int(
    profiles["transactions"]["columns"].set_index("Column").loc["batch_id", "Missing total"]
)
# The released child IDs are all valid. If that changes, count unmatched rows in a projected scan.
unmatched_ids = referenced_batch_ids - parent_ids
unmatched_rows = 0
if unmatched_ids:
    with pq.ParquetFile(paths["transactions"]) as parquet:
        bad_ids = pa.array(list(unmatched_ids), type=parquet.schema_arrow.field("batch_id").type)
        for block in parquet.iter_batches(batch_size=batch_size, columns=["batch_id"]):
            unmatched_rows += count_true(pc.is_in(block["batch_id"], value_set=bad_ids))
relationship_rows.append((
    "transactions.batch_id", "batches.batch_id", "many-to-one",
    transaction_row_count, missing_batch_ids, unmatched_rows, len(parent_ids - referenced_batch_ids),
))
relationships = pd.DataFrame(relationship_rows, columns=[
    "Child", "Parent", "Relationship", "Child rows", "Null references",
    "Unmatched non-null references", "Unreferenced parent IDs",
])
display(relationships)
assert relationships[["Null references", "Unmatched non-null references"]].eq(0).all().all()
assert relationships.loc[relationships["Relationship"].eq("one-to-one"), "Unreferenced parent IDs"].eq(0).all()

roots, matched_hashes, unmatched_hashes = ledger_counts
print(f"Hash link check: {roots:,} all-zero previous hashes, {matched_hashes:,} matching nonzero previous hashes, "
      f"{unmatched_hashes:,} unmatched nonzero previous hashes.")

transaction_schema = profiles["transactions"]["columns"].set_index("Column")
assert target in transaction_schema.index, f"Expected target is missing: {target}"
target_type = transaction_schema.loc[target, "Stored type"]
target_missing = int(transaction_schema.loc[target, "Missing total"])
assert target_type == "bool" and target_missing == 0
print(f"Target {target}: present, {target_type}, {target_missing:,} missing.")

## Save the schema summary

Write the results to `reports/schema_summary.md`. The report uses the counts and ranges computed above.

In [ ]:
def markdown_table(frame):
    """Format the small inspection tables without an extra dependency."""
    def format_value(value):
        if pd.isna(value):
            return ""
        if isinstance(value, (float, np.floating)):
            return format(value, ",.6g")
        if isinstance(value, (int, np.integer)):
            return format(value, ",d")
        return str(value).replace("|", r"\|")

    header = list(frame.columns)
    lines = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
    lines.extend("| " + " | ".join(map(format_value, row)) + " |" for row in frame.itertuples(index=False, name=None))
    return "\n".join(lines)


shapes = pd.DataFrame([
    (name, result["rows"], len(result["columns"]), duplicate_rows[name])
    for name, result in profiles.items()
], columns=["Table", "Rows", "Columns", "Duplicate records"])

report = [
    "# TRACE-X schema summary",
    "This structural review covers the five released Parquet files in `Phase1_Dataset/dataset/`. "
    "The [data understanding notebook](../notebooks/01_Data_Understanding.ipynb) shows the first five rows "
    "of each table when run locally and contains the checks used to produce this report. The expected keys and target came from the "
    "[dataset understanding report](dataset_understanding.md) and were checked against the release.",
    "## Shapes and duplicate records",
    markdown_table(shapes),
    "The four smaller tables were checked for exact full-row duplicates. The transactions file has zero "
    "full-row duplicates because its verified composite key has a different value on every row. The notebook "
    "reads at most 50,000 transaction rows per batch and keeps compact arrays for exact whole-file key checks.",
    "## Keys and relationships",
    "The verified logical primary keys are `entities.entity_id`, `medicines.medicine_id`, `batches.batch_id`, "
    "and `fraud_events.batch_id`. Transactions has no separate transaction ID. "
    "`(batch_id, leg, event_type)` identifies a lifecycle event, and `(batch_id, batch_tx_index)` is an alternate key. "
    "Both combinations are complete and unique in this release. These are observed keys in the data.",
    markdown_table(relationships),
    f"The ledger has {roots:,} all-zero previous hashes, {matched_hashes:,} nonzero previous hashes that point "
    f"to a stored current hash, and {unmatched_hashes:,} nonzero previous hashes without a match. "
    "The counts describe hash references only. They do not verify the order or integrity of the ledger.",
    "### Identifier counts",
    markdown_table(identifiers),
    "Distinct counts exclude missing values. Repeated rows count each additional occurrence of an identifier.",
    "## Target column",
    f"`transactions.{target}` is present as a Boolean column with {target_missing:,} missing values. "
    "It is the documented transaction-level target. This check covers presence, type, and completeness.",
    "## Column types, missing values, and ranges",
    "Types are the types read from Parquet. Missing values count nulls and floating-point NaNs separately. "
    "Numerical summaries use finite values and report the sample standard deviation. Dates and timestamps "
    "are stored as text in this release. Date ranges come from an in-memory format check. The event "
    "timestamp does not specify a timezone.",
]

for name, profile in profiles.items():
    report.extend([f"### {name}", markdown_table(profile["columns"])])
    if profile["numeric"].empty:
        report.append("This table has no numerical columns.")
    else:
        report.extend(["Numerical summary", markdown_table(profile["numeric"])])
    if profile["dates"].empty:
        report.append("This table has no date or timestamp columns.")
    else:
        report.extend(["Date or timestamp ranges", markdown_table(profile["dates"])])

report.append(
    "The original Parquet files were opened read-only. No datasets were cleaned, merged, or written. "
    "No feature-target relationships or preprocessing parameters were evaluated."
)
report_path = project_root / "reports" / "schema_summary.md"
report_path.write_text("\n\n".join(report) + "\n", encoding="utf-8")
print(f"Saved {report_path.relative_to(project_root)}")